In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os
import seaborn as sns
from scipy import stats as scipystats

# ***import file saved in script2*** 

In [3]:
# import data 

df = pd.read_csv(r"C:\Users\timon\OneDrive\Helmholtz\Scripts\Yagya_jupyter\dfs_for_final_code\optimised4MLR_SCD_SCGE_merged_dataset.csv")
df['Birth_or_div_vol_fl'] = df['Birth vol fl'].combine_first(df['Div vol fl'])

df.rename(columns={"abs_growth_rate_flpermin_inclIncompleteCycles": "abs_growth_rate"}, inplace= True)

# ***get relatives' volumes at each frame and calculate volume related parameters*** 

In [5]:
# add volumes of related cells at every frame 
new =[]
import warnings

with warnings.catch_warnings(record=True):
    for date in df['date'].unique():
        date_df  = df[df['date'] == date]
        for pos in date_df['Position'].unique():
            pos_df = date_df[date_df['Position'] == pos]
            for frame in pos_df['frame_i'].unique():
                frame_df = pos_df[pos_df['frame_i'] == frame]
                for cell in (frame_df['Cell_ID']).unique():


                    cell_ind = (frame_df[frame_df['Cell_ID'] == cell]).index
                    if frame_df['cell_cycle_stage'].loc[cell_ind[0]] == "S":

                        rel = frame_df['relative_ID'].loc[cell_ind[0]]

                        rel_ind = (frame_df[frame_df['Cell_ID'] == rel]).index
                        if len(rel_ind) == 0:
                            frame_df.loc[cell_ind[0],'rel_vol_atframe'] = np.nan
                            frame_df.loc[cell_ind[0],'sys_vol_atframe'] = np.nan
                            frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_sys'] = np.nan
                            frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_mother'] = np.nan

                        else:

                            cell_size_atframe = frame_df['cell_vol_fl'].loc[cell_ind[0]]
                            rel_size_atframe = frame_df['cell_vol_fl'].loc[rel_ind[0]]
                            frame_df.loc[cell_ind[0],'rel_vol_atframe'] = rel_size_atframe
                            frame_df.loc[cell_ind[0],'sys_vol_atframe'] = cell_size_atframe + rel_size_atframe
                            frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_mother'] = cell_size_atframe - (frame_df.loc[cell_ind[0],'mother_size_emerg'])
                            frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_sys'] = (frame_df.loc[cell_ind[0],'sys_vol_atframe']) - (frame_df.loc[cell_ind[0],'sys_size_emerg'])
                            frame_df.loc[cell_ind[0],'norm_delta_vol_sincePhaseStart_mother'] = (frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_mother']) / (frame_df.loc[cell_ind[0],'mother_size_emerg'])
                            frame_df.loc[cell_ind[0],'norm_delta_vol_sincePhaseStart_sys'] = (frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_sys']) / (frame_df.loc[cell_ind[0],'sys_size_emerg'])
                            frame_df.loc[cell_ind[0],'bud_vol_atframe/mother_vol_atframe'] = (frame_df.loc[cell_ind[0],'rel_vol_atframe']) / (frame_df.loc[cell_ind[0],'cell_vol_fl'])



                    else:
                        frame_df.loc[cell_ind[0],'rel_vol_atframe'] = np.nan
                        frame_df.loc[cell_ind[0],'sys_vol_atframe'] = np.nan
                        frame_df.loc[cell_ind[0],'delta_vol_sinceG1Start'] = (frame_df.loc[cell_ind[0],'cell_vol_fl']) - (frame_df.loc[cell_ind[0],'Birth_or_div_vol_fl'])
                        frame_df.loc[cell_ind[0],'norm_delta_vol_sinceG1Start'] = (frame_df.loc[cell_ind[0],'delta_vol_sinceG1Start'])/(frame_df.loc[cell_ind[0],'Birth_or_div_vol_fl'])

                    # for stage in frame_df['cell_cycle_stage'].unique():
                    #     stage_df = cycle_df[cycle_df['cell_cycle_stage'] == stage]
                    #
                    #     seconds_in_current_stage =

                new.append(frame_df)



new_df = pd.concat(new).reset_index(drop = True)
new_df = new_df.sort_values(["Unnamed: 0"])
#new_df.drop(columns=['Unnamed: 0.1.1.1','Unnamed: 0', 'Unnamed: 0.1.1.1.1'], axis = 1, inplace = True)
display(new_df.head(5))


,Unnamed: 0,generation_num,Position,Cell_ID,growthmedium,frame_i,cell_cycle_stage,relative_ID,relationship,cell_vol_fl,...,Birth_or_div_vol_fl,rel_vol_atframe,sys_vol_atframe,delta_vol_sincePhaseStart_mother,delta_vol_sincePhaseStart_sys,norm_delta_vol_sincePhaseStart_mother,norm_delta_vol_sincePhaseStart_sys,bud_vol_atframe/mother_vol_atframe,delta_vol_sinceG1Start,norm_delta_vol_sinceG1Start
0,0,0,Position_1,1,SCD,0,S,4,bud,58.256695,...,NaN,92.927348,151.184043,NaN,NaN,NaN,NaN,1.595136,NaN,NaN
8,1,0,Position_1,1,SCD,1,S,4,bud,60.404339,...,NaN,93.706004,154.110342,NaN,NaN,NaN,NaN,1.551312,NaN,NaN
16,2,0,Position_1,1,SCD,2,S,4,bud,61.139191,...,NaN,94.112864,155.252054,NaN,NaN,NaN,NaN,1.539321,NaN,NaN
24,3,0,Position_1,1,SCD,3,S,4,bud,62.500215,...,NaN,95.679685,158.179900,NaN,NaN,NaN,NaN,1.530870,NaN,NaN
32,4,0,Position_1,1,SCD,4,S,4,bud,63.211776,...,NaN,97.265445,160.477221,NaN,NaN,NaN,NaN,1.538723,NaN,NaN


# ***absolute growth rate calculation for cycles with complete G1 but incomplete S*** 

In [8]:
# absolute growth rate calculation for cycles with complete G1 but incomplete S 
new1 = []
for date in new_df['date'].unique():
    date_df  = new_df[new_df['date'] == date]
    for pos in date_df['Position'].unique():
        pos_df = date_df[date_df['Position'] == pos]
        for cell in pos_df['Cell_ID'].unique():
            cell_df = pos_df[pos_df['Cell_ID'] == cell]
            for cycle in cell_df['generation_num'].unique():
                cycle_df =  cell_df[cell_df['generation_num'] == cycle]


                frame_min = cycle_df['frame_i'].min()
                min_frame_index = cycle_df[cycle_df['frame_i'] == frame_min].index
                vol_min = cycle_df['cell_vol_fl'].loc[min_frame_index[0],]

                frame_max = cycle_df['frame_i'].max()
                max_frame_index = cycle_df[cycle_df['frame_i'] == frame_max].index

                if ((cycle_df['cell_cycle_stage'].loc[max_frame_index[0],] == "S") &
                    (np.isnan(cycle_df['sys_size_div'].loc[max_frame_index[0],]) == True) &
                    (cycle_df['generation_num'].loc[max_frame_index[0],] != 0)):
                    vol_max = cycle_df['sys_vol_atframe'].loc[max_frame_index[0],]

                    delta_vol_fl = vol_max-vol_min
                    delta_time_mins = (frame_max - frame_min)*3
                    abs_growth_rate_flpermin = delta_vol_fl/delta_time_mins

                    cycle_df['delta_vol_fl'] = delta_vol_fl
                    cycle_df['delta_time_mins'] = delta_time_mins
                    cycle_df['abs_growth_rate'] = abs_growth_rate_flpermin





                new1.append(cycle_df)

new1_df = pd.concat(new1).reset_index(drop = True)

new1_df.drop(columns=['Unnamed: 0'], inplace = True)

#display(new1_df.head(50))

C:\Users\timon\AppData\Local\Temp\ipykernel_8540\2727145400.py:27: RuntimeWarning: divide by zero encountered in scalar divide
  abs_growth_rate_flpermin = delta_vol_fl/delta_time_mins
C:\Users\timon\AppData\Local\Temp\ipykernel_8540\2727145400.py:27: RuntimeWarning: divide by zero encountered in scalar divide
  abs_growth_rate_flpermin = delta_vol_fl/delta_time_mins
C:\Users\timon\AppData\Local\Temp\ipykernel_8540\2727145400.py:27: RuntimeWarning: divide by zero encountered in scalar divide
  abs_growth_rate_flpermin = delta_vol_fl/delta_time_mins
C:\Users\timon\AppData\Local\Temp\ipykernel_8540\2727145400.py:27: RuntimeWarning: divide by zero encountered in scalar divide
  abs_growth_rate_flpermin = delta_vol_fl/delta_time_mins
C:\Users\timon\AppData\Local\Temp\ipykernel_8540\2727145400.py:27: RuntimeWarning: divide by zero encountered in scalar divide
  abs_growth_rate_flpermin = delta_vol_fl/delta_time_mins
C:\Users\timon\AppData\Local\Temp\ipykernel_8540\2727145400.py:27: RuntimeW

# ***assign daughter or not and calculate time in stage (mins)*** 
# ***save df*** 

In [13]:
############################################################ binary daughter or not #################################################################################
new1_df['Daughterhood_binary'] = 0
for cycle in new1_df['generation_num'].unique():
    ind = (new1_df[new1_df['generation_num'] == cycle]).index
    if cycle == 1:
        new1_df.loc[ind,'Daughterhood_binary'] = 1
#display(new1_df.head(100))

new2 = []
import warnings

with warnings.catch_warnings(record=True):
    
    for date in new1_df['date'].unique():
        date_df  = new1_df[new1_df['date'] == date]
        for pos in date_df['Position'].unique():
            pos_df = date_df[date_df['Position'] == pos]
            for cell in pos_df['Cell_ID'].unique():
                cell_df = pos_df[pos_df['Cell_ID'] == cell]
                for cycle in cell_df['generation_num'].unique():
                    cycle_df =  cell_df[cell_df['generation_num'] == cycle]
                    for stage in cycle_df['cell_cycle_stage'].unique():
                        stage_df = cycle_df[cycle_df['cell_cycle_stage'] == stage]
                        frame_at_stage_entry = stage_df['frame_i'].min()
                        for ind in stage_df.index:
                            stage_df.loc[ind,'time_in_stage_mins'] = ((stage_df['frame_i'].loc[ind]) - (frame_at_stage_entry))*3

                        new2.append(stage_df)



new2_df = pd.concat(new2).reset_index(drop = True)
#display(new2_df.head(100))

new2_df.to_csv(r"C:\Users\timon\OneDrive\Helmholtz\Scripts\Yagya_jupyter\dfs_for_final_code\optimised4MLR_SCD_SCGE_merged_dataset.csv")


Daughterhood_binary
0    239479
1    110630
Name: count, dtype: int64
